# Experiment 1 Supplement: Retrieval Analysis（实验一补充：检索分析）

## Install Packages（安装依赖）

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display
from retrieval_extensions import run, ROOT
DATA_DIR = ROOT / 'legal_faq_results'
RESULT_DIR = ROOT / 'retrieval_supplement'

## Evaluation Protocol and Criteria（评价协议与标准）

- single_gold：仅原始 doc_id 为相关文档。
- exact_answer_multi_gold：答案文本完全相同的 doc_id 均为相关文档；Recall@k = 前 k 位相关文档数 / 全部相关文档数，Hit@5 = 前五位是否至少命中一个，MRR = 首个相关文档排名的倒数，nDCG@5 以全部相关文档计算理想 DCG。
- duplicate_gold_queries_excluded：排除 gold 答案重复的查询，保留完整语料。
- 95%置信区间：以 FAQ/doc_id 为单位进行 2,000 次配对 bootstrap，五种查询变体作为同一组。

In [2]:
summary = run(DATA_DIR, RESULT_DIR, repeats=2000)
display(summary.round(4))

Duplicate sensitivity: BM25


Duplicate sensitivity: Dense


Duplicate sensitivity: Hybrid


,protocol,method,Recall@1,Recall@5,Recall@10,Hit@5,MRR,nDCG@5,query_type
0,duplicate_gold_queries_excluded,BM25,0.4640,0.6384,0.6929,0.6384,0.5452,0.5583,overall
1,duplicate_gold_queries_excluded,Dense,0.7448,0.9190,0.9482,0.9190,0.8225,0.8423,overall
2,duplicate_gold_queries_excluded,Hybrid,0.6011,0.7809,0.8392,0.7809,0.6873,0.6993,overall
3,exact_answer_multi_gold,BM25,0.4517,0.6332,0.6895,0.6344,0.5400,0.5535,overall
4,exact_answer_multi_gold,Dense,0.7237,0.9063,0.9383,0.9076,0.8098,0.8298,overall
5,exact_answer_multi_gold,Hybrid,0.5855,0.7743,0.8358,0.7763,0.6816,0.6932,overall
6,single_gold,BM25,0.4514,0.6332,0.6894,0.6332,0.5355,0.5495,overall
7,single_gold,Dense,0.7255,0.9062,0.9378,0.9062,0.8061,0.8263,overall
8,single_gold,Hybrid,0.5869,0.7742,0.8355,0.7742,0.6761,0.6889,overall


In [3]:
intervals = pd.read_csv(RESULT_DIR / 'bootstrap_ci.csv')
contrasts = pd.read_csv(RESULT_DIR / 'paired_differences_ci.csv')
display(intervals.query("protocol == 'single_gold' and query_type == 'overall'").round(4))
display(contrasts.query("protocol == 'single_gold' and query_type == 'overall'").round(4))

,query_type,metric,method,estimate,ci_low,ci_high,n_groups,bootstrap_repeats,protocol
216,overall,Recall@1,BM25,0.4514,0.4396,0.4637,2090,2000,single_gold
217,overall,Recall@1,Dense,0.7255,0.7123,0.7384,2090,2000,single_gold
218,overall,Recall@1,Hybrid,0.5869,0.5742,0.5990,2090,2000,single_gold
219,overall,Recall@5,BM25,0.6332,0.6214,0.6447,2090,2000,single_gold
220,overall,Recall@5,Dense,0.9062,0.8978,0.9153,2090,2000,single_gold
221,overall,Recall@5,Hybrid,0.7742,0.7640,0.7845,2090,2000,single_gold
222,overall,Recall@10,BM25,0.6894,0.6782,0.7010,2090,2000,single_gold
223,overall,Recall@10,Dense,0.9378,0.9308,0.9448,2090,2000,single_gold
224,overall,Recall@10,Hybrid,0.8355,0.8264,0.8445,2090,2000,single_gold
225,overall,Hit@5,BM25,0.6332,0.6214,0.6447,2090,2000,single_gold


,query_type,metric,contrast,difference,ci_low,ci_high,n_groups,protocol
216,overall,Recall@1,Dense - BM25,0.2741,0.2617,0.2862,2090,single_gold
217,overall,Recall@1,Hybrid - BM25,0.1355,0.1271,0.1438,2090,single_gold
218,overall,Recall@1,Hybrid - Dense,-0.1386,-0.1487,-0.1282,2090,single_gold
219,overall,Recall@5,Dense - BM25,0.2730,0.2620,0.2849,2090,single_gold
220,overall,Recall@5,Hybrid - BM25,0.1410,0.1336,0.1490,2090,single_gold
221,overall,Recall@5,Hybrid - Dense,-0.1321,-0.1414,-0.1231,2090,single_gold
222,overall,Recall@10,Dense - BM25,0.2484,0.2370,0.2592,2090,single_gold
223,overall,Recall@10,Hybrid - BM25,0.1461,0.1387,0.1537,2090,single_gold
224,overall,Recall@10,Hybrid - Dense,-0.1023,-0.1106,-0.0940,2090,single_gold
225,overall,Hit@5,Dense - BM25,0.2730,0.2620,0.2849,2090,single_gold


## Reformulation Audit Criteria（改写审核标准）

- meaning_preserved_yes_no_uncertain：yes = 核心意图与适用条件不变；no = 核心含义改变；uncertain = 无法确定。
- semantic_drift：yes = 对象、条件、范围或主题出现实质变化；no = 未发现实质变化。
- invalid：yes = 无法作为对应问题使用；no = 可以作为对应问题使用。

In [4]:
display(pd.read_csv(RESULT_DIR / 'query_quality_audit_100.csv').head())

,query_id,doc_id,query_type,query,original_question,reference_answer,meaning_preserved_yes_no_uncertain,semantic_drift,invalid,notes
0,Q004157,1747,formal,What is the application of proportionality in ...,How does the principle of proportionality appl...,Principle of Proportionality:\nThe principle o...,yes,no,no,NaN
1,Q007737,51,formal,What legal protections do I have concerning a ...,What are my rights regarding medical examinati...,Medical Examination after Arrest:\nYou have th...,yes,no,no,NaN
2,Q002762,1496,formal,Under what circumstances may Tamil Nadu Highwa...,What are the service conditions for Tamil Nadu...,Tamil Nadu Highway Patrol Service Conditions:\...,no,yes,no,改写问“在何种情况下可执勤”，原问题问服务条件（班次、津贴福利），参考答案不再对应。
3,Q003507,163,formal,How can one report instances of reckless driving?,What is the procedure to report reckless driving?,Reporting Reckless Driving:\nTo report reckles...,yes,no,no,NaN
4,Q002082,1373,formal,What processes are involved in recruiting poli...,How does police recruitment work for tribal yo...,Police Recruitment for Tribal Youth:\nThe Tami...,yes,no,no,NaN


In [5]:
from review_audit import audit_reformulations
audit_summary, audit_status = audit_reformulations()
display(pd.Series(audit_status))
display(audit_summary)
display(pd.read_csv(RESULT_DIR / 'query_quality_audit_issues.csv'))

n_sample                                                      100
n_completed_labels                                            100
n_issues                                                        2
status                 labels_complete_pending_consistency_review
annotations_changed                                         False
sample_seed                                                    42
per_type                                                       25
dtype: object

,query_type,field,label,count
0,casual,invalid,no,24
1,casual,invalid,yes,1
2,casual,meaning_preserved_yes_no_uncertain,no,1
3,casual,meaning_preserved_yes_no_uncertain,uncertain,5
4,casual,meaning_preserved_yes_no_uncertain,yes,19
5,casual,semantic_drift,no,20
6,casual,semantic_drift,yes,5
7,formal,invalid,no,25
8,formal,meaning_preserved_yes_no_uncertain,no,1
9,formal,meaning_preserved_yes_no_uncertain,uncertain,1


,query_id,query_type,issue,status
0,Q006714,search,preserved=yes conflicts with substantive drift...,researcher_confirmation_required
1,Q008885,hard,preserved=yes conflicts with substantive drift...,researcher_confirmation_required
